In [1]:
import modules.data as d
import modules.model as m
import modules.train2 as t
import modules.utils as u

import torch.nn as nn
from pathlib import Path

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

*** Device() ***
device = cuda:3

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:3)
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:3)
y                (567, 6)         Tensor (cuda:3)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



In [6]:
# get configs
model_config = t.Config(name='model')
model_config.add(
    key='classifier',
    model_class=m.MLPClassifier,
    model_kwargs={
        'in_features':data.num_nodes,
        'out_features':data.num_labels,
        'flatten':True,
    },
    trainer_class=t.Trainer,
    trainer_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'verbose':False,
    }
)

# define experiment, run experiment
exp = t.Experiment(
    num_trials=3,
    num_epochs=10,
    X=data.X,
    y=data.y,
    generator=generator,
)
exp.add_config(model_config)
exp.run_experiment(comment='training_modules_test2', verbose=True)

100%|██████████| 3/3 [00:00<00:00,  3.20it/s]


In [4]:
exp.summary['classifier']

,config,metric,mean,std,ci
0,model,loss,5.134147,3.44831,8.566078
